# 06 — Coherence Experiments

## Purpose

Measure the qubit's coherence times — $T_1$ (energy relaxation), $T_2^*$ (Ramsey dephasing), and $T_{2,\text{echo}}$ (Hahn echo) — to validate the current operating point and establish the coherence baseline for the calibrated qubit.

## Methodology

1. **T₁ relaxation** — excite the qubit with a π-pulse, wait a variable delay, then measure. The exponential decay of the excited-state population yields $T_1$.
2. **T₂ Ramsey** — apply π/2 – wait – π/2 with a small intentional detuning. The decaying oscillation gives $T_2^*$ and a frequency correction for the qubit drive.
3. **T₂ Echo (Hahn)** — apply π/2 – wait/2 – π – wait/2 – π/2 to refocus static dephasing. The envelope decay gives $T_{2,\text{echo}}$, an upper bound on useful coherence.

Each measurement is independently gated by a fit-quality threshold (r² > 0.5) and calibration patches are only committed when explicitly enabled.

## Expected Outcomes

- $T_1$ in the typical range for the device (10–200 µs for 3D transmons)
- $T_2^*$ showing Ramsey fringes with a clean sinusoidal envelope
- $T_{2,\text{echo}} \geq T_2^*$, confirming refocusing works
- If the Ramsey frequency correction is large (> 1 MHz), the qubit frequency may need re-calibration in notebook 05

## Prerequisites

- **Notebook 05** — qubit frequency and π-pulse calibrated

## 1. Setup — Session Bootstrap

Open the notebook stage and load the prior pulse calibration checkpoint from notebook 05.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    T1Relaxation,
    T2Echo,
    T2Ramsey,
    ensure_primitive_rotations,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2026_03_24"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="06_coherence_experiments",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)
INITIAL_QB_FQ_HZ = float(attr.qb_fq)

pulse_calibration_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="05_qubit_spectroscopy_pulse_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"Closed a live in-memory session before reopen: {stage.had_live_session}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current qubit frequency: {float(attr.qb_fq) / 1e9:.6f} GHz")
if pulse_calibration_checkpoint is not None:
    print(
        "Prior stage 05 status: "
        f"{pulse_calibration_checkpoint['status']}"
        f" ({pulse_calibration_checkpoint['summary']})"
    )
    persisted = pulse_calibration_checkpoint.get("persisted_outputs", {})
    print(
        "Stage 05 applied flags: "
        f"qubit={persisted.get('qubit_frequency_applied', False)}, "
        f"power_rabi={persisted.get('power_rabi_applied', False)}, "
        f"temporal_rabi={persisted.get('temporal_rabi_applied', False)}"
    )

2026-03-22 00:53:38,029 - qm - INFO     - Starting session: 33aba75f-c06f-48a2-8cd8-23c8d0bd5f25
[INFO] 2026-03-22 00:53:39,324 qubox.legacy.devices.context_resolver: Resolved context: sample=post_cavity_sample_A cooldown=cd_2025_02_22 wiring=8b82c977
[INFO] 2026-03-22 00:53:39,325 qubox.legacy.experiments.session: Context mode: sample=post_cavity_sample_A cooldown=cd_2025_02_22 wiring=8b82c977
[INFO] 2026-03-22 00:53:39,326 qubox.legacy.experiments.session: SessionManager initialising at E:\qubox\samples\post_cavity_sample_A\cooldowns\cd_2025_02_22
[INFO] 2026-03-22 00:53:39,327 qubox.legacy.hardware.config_engine: Hardware loaded from E:\qubox\samples\post_cavity_sample_A\config\hardware.json
2026-03-22 00:53:42,701 - qm - INFO     - Performing health check
2026-03-22 00:53:42,706 - qm - INFO     - Health check passed
[INFO] 2026-03-22 00:53:42,707 qubox.legacy.pulses.manager: Loaded pulse files from: E:\qubox\samples\post_cavity_sample_A\cooldowns\cd_2025_02_22\config\pulses.json
[I

## 2. Configuration — Coherence Measurement Defaults

Define delay ranges, averaging counts, fit quality thresholds, and calibration-apply flags for each coherence experiment. The Ramsey detuning of 0.2 MHz creates visible fringes without broadening the effective envelope too much.

In [ ]:
APPLY_T1_CALIBRATION = False
APPLY_T2_RAMSEY_CALIBRATION = True
APPLY_T2_RAMSEY_FREQUENCY_CORRECTION = True
APPLY_T2_ECHO_CALIBRATION = False

QB_THERM_CLKS = int(getattr(attr, "qb_therm_clks", 10000) or 10000)

REF_R180_LEN_NS = 16
REF_R180_AMPLITUDE = 0.08565235748770193
REF_R180_SIGMA_NS = REF_R180_LEN_NS / 6
REF_R180_ALPHA = 0.0
REF_R180_ANHARMONICITY_HZ = float(getattr(attr, "anharmonicity", -255e6) or -255e6)
REF_R180_DETUNING_HZ = 0.0
REF_R180_SAMPLING_RATE = 1e9

T1_DELAY_END = 50 * u.us
T1_DT = 500
T1_N_AVG = 2000
T1_MIN_R_SQUARED = 0.5

T2_RAMSEY_QB_DETUNE_MHZ = 0.2
T2_RAMSEY_DELAY_END = 40 * u.us
T2_RAMSEY_DT = 100
T2_RAMSEY_N_AVG = 2000
T2_RAMSEY_MIN_R_SQUARED = 0.5
T2_RAMSEY_MAX_FREQ_CORRECTION_MHZ = 2.0

T2_ECHO_DELAY_END = 40 * u.us
T2_ECHO_DT = 100
T2_ECHO_N_AVG = 2000
T2_ECHO_MIN_R_SQUARED = 0.5

print("Coherence verification settings:")
print(f"  apply T1 calibration: {APPLY_T1_CALIBRATION}")
print(f"  apply T2 Ramsey calibration: {APPLY_T2_RAMSEY_CALIBRATION}")
print(f"  apply T2 Ramsey frequency correction: {APPLY_T2_RAMSEY_FREQUENCY_CORRECTION}")
print(f"  apply T2 Echo calibration: {APPLY_T2_ECHO_CALIBRATION}")
print(f"  qb_therm_clks: {QB_THERM_CLKS}")
print(f"  T1 n_avg: {T1_N_AVG}")
print(f"  T2 Ramsey n_avg: {T2_RAMSEY_N_AVG}")
print(f"  T2 Echo n_avg: {T2_ECHO_N_AVG}")
print(f"  T2 Ramsey detune: {T2_RAMSEY_QB_DETUNE_MHZ:.3f} MHz")

## 3. Execution — T₁, Ramsey, Echo, and Checkpoint

Run all three coherence experiments in sequence, apply fit-quality gates, optionally commit calibration patches, and save the stage checkpoint.

In [ ]:
primitive_seed_result = ensure_primitive_rotations(
    session,
    qb_element=attr.qb_el,
    amplitude=float(REF_R180_AMPLITUDE),
    length=int(REF_R180_LEN_NS),
    sigma=float(REF_R180_SIGMA_NS),
    alpha=float(REF_R180_ALPHA),
    anharmonicity_hz=float(REF_R180_ANHARMONICITY_HZ),
    detuning_hz=float(REF_R180_DETUNING_HZ),
    sampling_rate=float(REF_R180_SAMPLING_RATE),
    required_ops=("x180", "x90"),
    rotations=("ref_r180", "x180", "x90", "xn90", "y180", "y90", "yn90"),
    ref_op="ref_r180",
)
if primitive_seed_result["created"]:
    print(f"Registered primitive rotations for notebook 06: {primitive_seed_result['created_ops']}")
else:
    print("Primitive rotations already available for notebook 06.")


t1 = T1Relaxation(session)
t1_result = t1.run(
    delay_end=T1_DELAY_END,
    dt=T1_DT,
    n_avg=T1_N_AVG,
    use_circuit_runner=False,
)
t1_analysis = t1.analyze(
    t1_result,
    update_calibration=True,
    derive_qb_therm_clks=True,
)
t1.plot(t1_analysis)
measured_t1_us = float(t1_analysis.metrics.get("T1_us", np.nan))
t1_fit_ok, t1_fit_reason = fit_quality_gate(t1_analysis, r_squared_min=T1_MIN_R_SQUARED)
if t1_fit_ok and (not np.isfinite(measured_t1_us) or measured_t1_us <= 0):
    t1_fit_ok = False
    t1_fit_reason = f"non-physical T1 fit: T1={measured_t1_us:.3f} us"
print(f"Measured T1: {measured_t1_us:.3f} us")
print(f"T1 fit gate: {t1_fit_reason}")
t1_patch = None
t1_patch_preview = None
t1_apply_result = None
if t1_fit_ok:
    t1_patch, t1_patch_preview, t1_apply_result = preview_or_apply_patch_ops(
        session,
        reason="T1 coherence calibration",
        proposed_patch_ops=t1_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_T1_CALIBRATION,
    )

t2_ramsey = T2Ramsey(session)
t2_ramsey_result = t2_ramsey.run(
    qb_detune=int(T2_RAMSEY_QB_DETUNE_MHZ * 1e6),
    delay_end=T2_RAMSEY_DELAY_END,
    dt=T2_RAMSEY_DT,
    n_avg=T2_RAMSEY_N_AVG,
    qb_therm_clks=QB_THERM_CLKS,
    qb_detune_MHz=T2_RAMSEY_QB_DETUNE_MHZ,
)
t2_ramsey_analysis = t2_ramsey.analyze(
    t2_ramsey_result,
    update_calibration=True,
    apply_frequency_correction=APPLY_T2_RAMSEY_FREQUENCY_CORRECTION,
)
t2_ramsey.plot(t2_ramsey_analysis)
measured_t2_star_us = float(
    t2_ramsey_analysis.metrics.get(
        "T2_star_us",
        float(t2_ramsey_analysis.metrics.get("T2_star", np.nan)) / 1e3,
    )
)
ramsey_fit_ok, ramsey_fit_reason = fit_quality_gate(
    t2_ramsey_analysis,
    r_squared_min=T2_RAMSEY_MIN_R_SQUARED,
)
if ramsey_fit_ok and (not np.isfinite(measured_t2_star_us) or measured_t2_star_us <= 0):
    ramsey_fit_ok = False
    ramsey_fit_reason = f"non-physical Ramsey fit: T2*={measured_t2_star_us:.3f} us"
qb_freq_correction_hz = float(t2_ramsey_analysis.metrics.get("qb_freq_correction_Hz", np.nan))
if ramsey_fit_ok and np.isfinite(qb_freq_correction_hz):
    if abs(qb_freq_correction_hz) > T2_RAMSEY_MAX_FREQ_CORRECTION_MHZ * 1e6:
        ramsey_fit_ok = False
        ramsey_fit_reason = (
            f"Ramsey frequency correction exceeds limit: {qb_freq_correction_hz / 1e6:+.3f} MHz "
            f"> {T2_RAMSEY_MAX_FREQ_CORRECTION_MHZ:.3f} MHz"
        )
print(f"Measured T2 Ramsey: {measured_t2_star_us:.3f} us")
print(f"Ramsey fit gate: {ramsey_fit_reason}")
if np.isfinite(qb_freq_correction_hz):
    print(f"Ramsey qubit frequency correction: {qb_freq_correction_hz / 1e6:+.6f} MHz")
    print(f"Ramsey corrected qubit frequency: {float(t2_ramsey_analysis.metrics.get('qb_freq_corrected_Hz', np.nan)) / 1e9:.6f} GHz")
t2_ramsey_patch = None
t2_ramsey_patch_preview = None
t2_ramsey_apply_result = None
if ramsey_fit_ok:
    t2_ramsey_patch, t2_ramsey_patch_preview, t2_ramsey_apply_result = preview_or_apply_patch_ops(
        session,
        reason="T2 Ramsey coherence and frequency calibration",
        proposed_patch_ops=t2_ramsey_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_T2_RAMSEY_CALIBRATION,
    )

t2_echo = T2Echo(session)
t2_echo_result = t2_echo.run(
    delay_end=T2_ECHO_DELAY_END,
    dt=T2_ECHO_DT,
    n_avg=T2_ECHO_N_AVG,
)
t2_echo_analysis = t2_echo.analyze(t2_echo_result, update_calibration=True)
t2_echo.plot(t2_echo_analysis)
measured_t2_echo_us = float(
    t2_echo_analysis.metrics.get(
        "T2_echo_us",
        float(t2_echo_analysis.metrics.get("T2_echo", np.nan)) / 1e3,
    )
)
t2_echo_fit_ok, t2_echo_fit_reason = fit_quality_gate(
    t2_echo_analysis,
    r_squared_min=T2_ECHO_MIN_R_SQUARED,
)
if t2_echo_fit_ok and (not np.isfinite(measured_t2_echo_us) or measured_t2_echo_us <= 0):
    t2_echo_fit_ok = False
    t2_echo_fit_reason = f"non-physical T2 Echo fit: T2_echo={measured_t2_echo_us:.3f} us"
print(f"Measured T2 Echo: {measured_t2_echo_us:.3f} us")
print(f"T2 Echo fit gate: {t2_echo_fit_reason}")
t2_echo_patch = None
t2_echo_patch_preview = None
t2_echo_apply_result = None
if t2_echo_fit_ok:
    t2_echo_patch, t2_echo_patch_preview, t2_echo_apply_result = preview_or_apply_patch_ops(
        session,
        reason="T2 Echo coherence calibration",
        proposed_patch_ops=t2_echo_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_T2_ECHO_CALIBRATION,
    )

context_snapshot = getattr(session, "context_snapshot", None)
attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
if attr is not None:
    print(f"Runtime qubit frequency after notebook 06 runs: {float(attr.qb_fq) / 1e9:.6f} GHz")
    print(f"Runtime qb_therm_clks after notebook 06 runs: {int(getattr(attr, 'qb_therm_clks', 0) or 0)}")

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
    status="calibrated" if any(result is not None for result in (t1_apply_result, t2_ramsey_apply_result, t2_echo_apply_result)) else "validated",
    summary="Validated coherence fits and committed only the operator-approved coherence or frequency updates.",
    consumed_inputs={
        "t1_n_avg": T1_N_AVG,
        "t2_ramsey_n_avg": T2_RAMSEY_N_AVG,
        "t2_echo_n_avg": T2_ECHO_N_AVG,
        "qb_therm_clks": QB_THERM_CLKS,
    },
    persisted_outputs={
        "t1_applied": t1_apply_result is not None,
        "t2_ramsey_applied": t2_ramsey_apply_result is not None,
        "t2_echo_applied": t2_echo_apply_result is not None,
        "runtime_qb_fq_hz": float(attr.qb_fq) if attr is not None else np.nan,
        "runtime_qb_therm_clks": int(getattr(attr, "qb_therm_clks", 0) or 0) if attr is not None else 0,
    },
    advisory_outputs={
        "t1_us": measured_t1_us,
        "t2_star_us": measured_t2_star_us,
        "t2_echo_us": measured_t2_echo_us,
        "ramsey_qb_freq_correction_hz": qb_freq_correction_hz,
    },
    next_stage=None,
    notes=[
        "This stage is validation-first: fits must pass explicit quality gates before any calibration is eligible to commit.",
        "Frequency correction remains separately controllable through APPLY_T2_RAMSEY_FREQUENCY_CORRECTION and APPLY_T2_RAMSEY_CALIBRATION.",
    ],
    metrics={
        "t1": dict(t1_analysis.metrics),
        "t2_ramsey": dict(t2_ramsey_analysis.metrics),
        "t2_echo": dict(t2_echo_analysis.metrics),
    },
)
print(f"Stage checkpoint saved to: {stage_checkpoint_path}")

## 4. Analysis — Coherence Fit Overlays

Projected signal vs. delay plots with model overlays for each coherence experiment. Each panel shows the measured data points and the best-fit curve, annotated with the extracted coherence time and (for Ramsey) the frequency correction.

In [ ]:
from qubox_tools.fitting import cqed as cQED_models
from qubox_tools.algorithms.transforms import project_complex_to_line_real

plot_specs = []
if "t1_analysis" in globals():
    plot_specs.append(("T1", t1_analysis))
if "t2_ramsey_analysis" in globals():
    plot_specs.append(("T2 Ramsey", t2_ramsey_analysis))
if "t2_echo_analysis" in globals():
    plot_specs.append(("T2 Echo", t2_echo_analysis))

if not plot_specs:
    print("Run at least one coherence experiment above to populate the fit views.")
else:
    fig, axes = plt.subplots(1, len(plot_specs), figsize=(5.6 * len(plot_specs), 4.6), squeeze=False)
    axes = axes.ravel()
    summary_rows = []

    for ax, (label, analysis) in zip(axes, plot_specs):
        delays_ns = np.asarray(analysis.data.get("delays", []), dtype=float)
        projected = analysis.data.get("projected_S")
        if projected is None:
            S = np.asarray(analysis.data.get("S", []))
            projected, _, _ = project_complex_to_line_real(S)
        projected = np.asarray(projected, dtype=float)
        if delays_ns.size == 0 or projected.size == 0:
            ax.set_visible(False)
            continue

        ax.plot(delays_ns / 1e3, projected, "o", ms=4, color="#2f4b7c", label="Projected signal")

        fit_params = getattr(getattr(analysis, "fit", None), "params", None) or {}
        dense_delays_ns = np.linspace(float(delays_ns.min()), float(delays_ns.max()), 400)
        measured_us = np.nan

        if label == "T1" and {"A", "T1", "offset"}.issubset(fit_params):
            fit_trace = cQED_models.T1_relaxation_model(dense_delays_ns, fit_params["A"], fit_params["T1"], fit_params["offset"])
            measured_us = float(fit_params["T1"]) / 1e3
        elif label == "T2 Ramsey" and {"A", "T2", "n", "f_det", "phi", "offset"}.issubset(fit_params):
            fit_trace = cQED_models.T2_ramsey_model(
                dense_delays_ns,
                fit_params["A"],
                fit_params["T2"],
                fit_params["n"],
                fit_params["f_det"],
                fit_params["phi"],
                fit_params["offset"],
            )
            measured_us = float(fit_params["T2"]) / 1e3
        elif label == "T2 Echo" and {"A", "T2_echo", "n", "offset"}.issubset(fit_params):
            fit_trace = cQED_models.T2_echo_model(
                dense_delays_ns,
                fit_params["A"],
                fit_params["T2_echo"],
                fit_params["n"],
                fit_params["offset"],
            )
            measured_us = float(fit_params["T2_echo"]) / 1e3
        else:
            fit_trace = None

        if fit_trace is not None:
            ax.plot(dense_delays_ns / 1e3, fit_trace, "-", lw=2, color="#bc5090", label="Fit")

        text_lines = []
        if np.isfinite(measured_us):
            text_lines.append(f"measured = {measured_us:.3f} us")
        if label == "T2 Ramsey":
            freq_correction_hz = float(analysis.metrics.get("qb_freq_correction_Hz", np.nan))
            corrected_qb_hz = float(analysis.metrics.get("qb_freq_corrected_Hz", np.nan))
            if np.isfinite(freq_correction_hz):
                text_lines.append(f"dq = {freq_correction_hz / 1e6:+.3f} MHz")
            if np.isfinite(corrected_qb_hz):
                text_lines.append(f"qb_fq = {corrected_qb_hz / 1e9:.6f} GHz")

        if text_lines:
            ax.text(
                0.03,
                0.05,
                "\n".join(text_lines),
                transform=ax.transAxes,
                va="bottom",
                ha="left",
                fontsize=9,
                bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "#cccccc"},
            )

        summary_rows.append((label, measured_us))
        ax.set_title(label)
        ax.set_xlabel("Delay (us)")
        ax.set_ylabel("Projected S_I")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")

    plt.tight_layout()
    plt.show()

    for label, measured_us in summary_rows:
        if np.isfinite(measured_us):
            print(f"{label}: measured {measured_us:.3f} us")

    if "t2_ramsey_analysis" in globals():
        freq_correction_hz = float(t2_ramsey_analysis.metrics.get("qb_freq_correction_Hz", np.nan))
        corrected_qb_hz = float(t2_ramsey_analysis.metrics.get("qb_freq_corrected_Hz", np.nan))
        if np.isfinite(freq_correction_hz):
            print(f"Ramsey frequency correction: {freq_correction_hz / 1e6:+.6f} MHz")
        if np.isfinite(corrected_qb_hz):
            print(f"Ramsey corrected qubit frequency: {corrected_qb_hz / 1e9:.6f} GHz")
            print(f"Runtime delta from bootstrap: {(corrected_qb_hz - INITIAL_QB_FQ_HZ) / 1e6:+.6f} MHz")